# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10  | s["b"] берёт значение по ярлыку "b" (20), а s.iloc[0] — по нулевой позиции (10)
2. Series: Алматы NaN, Астана NaN, Шымкент 3.0 | Pandas складывает по совпадающим ярлыкам(городам) и сортирует их по алфавиту; для городов, которых нет в одной из Series, ставит NaN
3. (2,2) | По условию x > 1 остаются 2 строки (где x равен 2 и 3), а столбцов по-прежнему 2
4. Series DataFrame | Одни скобки df["x"] возвращают столбец (Series), а двойные df[["x"]] — таблицу (DataFrame)
5. [1,2] 5 | После фильтрации сохраняются старые индексы строк (счёт с 0, поэтому остаются ярлыки 1 и 2). Первая строка после фильтра — где x=2, y=5, столбец с индексом 1 — это y

In [4]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [7]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [7]:
# 1

df.shape

(5463, 8)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 671.2 KB


In [12]:
df[["city", "category", "status", "customer_id"]].nunique()

city             14
category          5
status            2
customer_id    2003
dtype: int64

В таблице 5 463 строки и 8 столбцов; в delivery_days 160 пропусков. Уникальных значений: 14 в city, 5 в category, 2 в status, 2003 в customer_id.
Три проблемы:
1. Разные написания городов (14 вместо 6): из-за пробелов, регистра и латиницы («Шымкент», «шымкент», «Shymkent») выручка одного города дробится, и статистика по филиалам занижается.
2. Смешанные статусы заказов (оплачен и отменён): если не фильтровать по статусу, в выручку попадут отменённые заказы, по которым магазин не получил денег.
3. Тестовый клиент TEST и возможные дубли (помимо текстового формата amount в сыром файле): технические заказы TEST с огромными суммами и повторяющиеся строки искусственно раздувают выручку и искажают динамику.

    

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [9]:
# 2
df.loc[paid & not_test, "amount"].describe()

count      4901.000000
mean      14672.391349
std       13249.340787
min         720.000000
25%        5880.000000
50%       10300.000000
75%       18690.000000
max      139040.000000
Name: amount, dtype: float64

Типичный чек медиана (10300), а половина всех заказов лежит в диапозоне от 5880 до 18690. Честнее указать в отчете медиану (10300), так как среднее  (14672) сильно завышено дорогими единичными заказами (до 139 040 ₸ в электронике)

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [11]:
# 3
SH = ["Шымкент", "шымкент", "Shymkent"]
df["city"].unique()

<ArrowStringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str

In [25]:
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
paid = df["status"] == "оплачен"
not_paid = ~paid
print(f"Доля отменённых заказов: {not_paid.mean():.3f}")
print(f"Доля отмененных заказов в Шымкенте: {not_paid[df["city"].isin(SH)].mean():.3f}")
print(f"Доля отмененных заказов в Алматы: {not_paid[df["city"].isin(AL)].mean():.3f}")

Доля отменённых заказов: 0.102
Доля отмененных заказов в Шымкенте: 0.204
Доля отмененных заказов в Алматы: 0.079


Выделяется Шымкент: доля отмен там (20,4%) в 2 раза выше средней по всему магазину (10,2%) и в 2,6 раза выше, чем в Алматы (7,9%).
Что делать менеджеру? — Проверить логистику и работу курьерской службы в Шымкенте.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [40]:
# 4
shym_sep = paid & not_test & sep & df["city"].isin(SH)
shym_oct = paid & not_test & octo & df["city"].isin(SH)
alm_sep = paid & not_test & sep & df["city"].isin(AL)
alm_oct = paid & not_test & octo & df["city"].isin(AL)
rev_shym_sep = df.loc[shym_sep, "amount"].sum()
rev_shym_oct = df.loc[shym_oct, "amount"].sum()
rev_alm_sep = df.loc[alm_sep, "amount"].sum()
rev_alm_oct = df.loc[alm_oct, "amount"].sum()
shym_total = (rev_shym_oct - rev_shym_sep)/rev_shym_sep * 100
alm_total = (rev_alm_oct - rev_alm_sep)/rev_alm_sep * 100
print(f"Выручка за сентябрь Шымкент: {rev_shym_sep}")
print(f"Выручка за октябрь Шымкент: {rev_shym_oct}")

print(f"Выручка за сентябрь Алматы: {rev_alm_sep}")
print(f"Выручка за октябрь Алматы: {rev_alm_oct}")

print(f"Относительные изменения Шымкент: {shym_total:.2f}%")
print(f"Относительные изменения Алматы: {alm_total:.2f}%")

Выручка за сентябрь Шымкент: 4949800.0
Выручка за октябрь Шымкент: 4044010.0
Выручка за сентябрь Алматы: 15523940.0
Выручка за октябрь Алматы: 14643020.0
Относительные изменения Шымкент: -18.30%
Относительные изменения Алматы: -5.67%


Падение сильнее в Шымкенте, примерно в 3,2 раза (18.3% / 5.67%). Общее проседание выручки в октябре тянет вниз Шымкент, падение на 18,3%.нужно срочно разбираться с доставкой/логистикой и причинами отказов в Шымкенте, тогда как в Алматы снижение умеренное -5,7%.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [45]:
# 5
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


1 ошибка не раставлены ( ) вокруг условий. 2 ошибка не берутся во внимания другие вариации написания города, нужно через метод .isin

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [63]:
df["customer_id"].value_counts()

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
          ..
C00645     1
C01442     1
C00018     1
C01650     1
C01057     1
Name: count, Length: 2003, dtype: int64

In [60]:
df["customer_id"].value_counts().idxmax()

'C01818'

In [61]:
mask = df["customer_id"] == "C01818"

In [62]:
df.loc[mask, ["order_id", "order_date", "city", "amount", "status"]]

,order_id,order_date,city,amount,status
57,101125,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,100096,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,101860,2026-09-24 18:00,Астана,1490.0,оплачен
2617,101462,2026-09-29 09:00,Астана,22170.0,оплачен
2776,101495,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,103416,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,102644,2026-10-18 08:00,Астана,13600.0,оплачен
4610,104001,2026-10-21 19:00,Алматы,21990.0,оплачен


Есть дубль заказа order_id = 101504, повторяется два раза. Есть большая вероятность ,что в таблице есть еще дубли и это искажает информацию об итоговой выручке в большую сторону.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8